In [ ]:
import re

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from shared import SOCIAL_CSV, CODED_CSV, THEMES

In [ ]:
df = pd.read_csv(SOCIAL_CSV, parse_dates=["date"])
len(df)

1083

In [ ]:
analyzer = SentimentIntensityAnalyzer()
df["vader"] = df["review_text"].apply(lambda s: analyzer.polarity_scores(s)["compound"])
df["vader"].describe()

count    1083.000000
mean        0.185661
std         0.547219
min        -0.962300
25%        -0.248400
50%         0.250000
75%         0.680800
max         0.994500
Name: vader, dtype: float64

In [ ]:
def sentiment_label(v):
    if v >= 0.05:
        return "positive"
    if v <= -0.05:
        return "negative"
    return "neutral"


df["sentiment"] = df["vader"].apply(sentiment_label)
df["sentiment"].value_counts()

sentiment
positive    599
negative    328
neutral     156
Name: count, dtype: int64

In [ ]:
text = df["review_text"].str.lower()

for theme, terms in THEMES.items():
    patterns = []
    for term in terms:
        if term == "red":
            patterns.append(r"\bred\b")
        elif re.match(r"^[a-z]", term):
            patterns.append(r"\b" + re.escape(term))
        else:
            patterns.append(re.escape(term))
    df[theme] = text.str.contains("|".join(patterns), regex=True).astype(int)

In [ ]:
(df[list(THEMES)].mean() * 100).sort_values(ascending=False).round(1)

scent              38.4
efficacy           36.7
ingredients        19.8
irritation          9.2
transition          9.0
price_value         8.2
ownership_trust     2.0
packaging           1.8
dtype: float64

In [ ]:
df.to_csv(CODED_CSV, index=False)